In [1]:
!pip install pytensor


In [2]:
import pytensor.tensor as T
from pytensor import function
a = T.dscalar('a')
b = T.dscalar('b')
c = T.dscalar('c')
d = T.dscalar('d')
e = T.dscalar('e')
f = ((a-b+c)*d)/e
g = function([a,b,c,d,e],f)
print("Expected: ((1 - 2 + 3) * 4)/5.0 = ", ((1 - 2 + 3) * 4)/5.0)
print("Via Theano: ((1 - 2 + 3) * 4)/5.0 = ", g(1, 2, 3, 4, 5))

Expected: ((1 - 2 + 3) * 4)/5.0 =  1.6
Via Theano: ((1 - 2 + 3) * 4)/5.0 =  1.6


In [6]:
import numpy

In [7]:
a = T.dmatrix('a')
b = T.dmatrix('b')
c = T.dmatrix('c')
d = T.dmatrix('d')

e = (a + b - c) * d
f = function([a,b,c,d], e)
a_data = numpy.array([[1,1],[1,1]])
b_data = numpy.array([[2,2],[2,2]])
c_data = numpy.array([[5,5],[5,5]])
d_data = numpy.array([[3,3],[3,3]])
print("Expected:", (a_data + b_data - c_data) * d_data)
print("Via Theano:", f(a_data,b_data,c_data,d_data))

Expected: [[-6 -6]
 [-6 -6]]
Via Theano: [[-6. -6.]
 [-6. -6.]]


In [8]:
a = T.dmatrix('a')
b = T.dmatrix('b')
c = T.dmatrix('c')
d = T.dmatrix('d')
p = T.dscalar('p')
q = T.dscalar('q')
r = T.dscalar('r')
s = T.dscalar('s')
u = T.dscalar('u')

e = (((a * p) + (b - q) - (c + r )) * d/s) * u

f = function([a,b,c,d,p,q,r,s,u], e)
a_data = numpy.array([[1,1],[1,1]])
b_data = numpy.array([[2,2],[2,2]])
c_data = numpy.array([[5,5],[5,5]])
d_data = numpy.array([[3,3],[3,3]])
print("Expected:", (((a_data * 1.0) + (b_data - 2.0) - (c_data + 3.0 )) * d_data/4.0) * 5.0)
print("Via Theano:", f(a_data,b_data,c_data,d_data,1,2,3,4,5))

Expected: [[-26.25 -26.25]
 [-26.25 -26.25]]
Via Theano: [[-26.25 -26.25]
 [-26.25 -26.25]]


# **Activation Functions**

In [18]:
import pytensor
import pytensor.tensor as T
from pytensor import function

# 1. Sigmoid - now directly in T
a = T.dmatrix('a')
f_a = T.sigmoid(a)
f_sigmoid = function([a], [f_a])
print("sigmoid:", f_sigmoid([[-1, 0, 1]]))

# 2. Tanh - directly in T
b = T.dmatrix('b')
f_b = T.tanh(b)
f_tanh = function([b], [f_b])
print("tanh:", f_tanh([[-1, 0, 1]]))

# 3. Fast Sigmoid - Raw Math implementation
c = T.dmatrix('c')
f_c = c / (1 + T.abs(c))
f_fast_sigmoid = function([c], [f_c])
print("fast sigmoid:", f_fast_sigmoid([[-1, 0, 1]]))

# 4. Softplus - now directly in T
d = T.dmatrix('d')
f_d = T.softplus(d)
f_softplus = function([d], [f_d])
print("soft plus:", f_softplus([[-1, 0, 1]]))

# 5. ReLU - Using the robust mathematical definition
e = T.dmatrix('e')
f_e = T.maximum(0, e)
f_relu = function([e], [f_e])
print("relu:", f_relu([[-1, 0, 1]]))

# 6. Softmax - Using a manual stable implementation
f_var = T.dmatrix('f_var')
e_x = T.exp(f_var - T.max(f_var, axis=1, keepdims=True))
f_f = e_x / e_x.sum(axis=1, keepdims=True)

f_softmax = function([f_var], [f_f])
print("soft max:", f_softmax([[-1, 0, 1]]))

sigmoid: [array([[0.26894142, 0.5       , 0.73105858]])]
tanh: [array([[-0.76159416,  0.        ,  0.76159416]])]
fast sigmoid: [array([[-0.5,  0. ,  0.5]])]
soft plus: [array([[0.31326169, 0.69314718, 1.31326169]])]
relu: [array([[0., 0., 1.]])]
soft max: [array([[0.09003057, 0.24472847, 0.66524096]])]


In [20]:
from pytensor import shared
x = T.dmatrix('x')
y = shared(numpy.array([[4, 5, 6]]))
z = x + y
f = function(inputs = [x], outputs = [z])
print("Original Shared Value:", y.get_value())
print("Original Function Evaluation:", f([[1, 2, 3]]))
y.set_value(numpy.array([[5, 6, 7]]))
print("Original Shared Value:", y.get_value())
print("Original Function Evaluation:", f([[1, 2, 3]]))

Original Shared Value: [[4 5 6]]
Original Function Evaluation: [array([[5., 7., 9.]])]
Original Shared Value: [[5 6 7]]
Original Function Evaluation: [array([[ 6.,  8., 10.]])]


In [21]:
x = T.dmatrix('x')
y = shared(numpy.array([[4, 5, 6]]))
z = T.sum(((x * x) + y) * x)
f = function(inputs = [x], outputs = [z])
g = T.grad(z,[x])
g_f = function([x], g)

print("Original:", f([[1, 2, 3]]))
print("Original Gradient:", g_f([[1, 2, 3]]))
y.set_value(numpy.array([[1, 1, 1]]))
print("Updated:", f([[1, 2, 3]]))
print("Updated Gradient", g_f([[1, 2, 3]]))

Original: [array(68.)]
Original Gradient: [array([[ 7., 17., 33.]])]
Updated: [array(42.)]
Updated Gradient [array([[ 4., 13., 28.]])]


In [26]:
import sklearn.metrics

def l2(x):
    return T.sum(x**2)
examples = 1000
features = 100
hidden = 10

D = (numpy.random.randn(examples, features), numpy.random.randint(size=examples,low=0, high=2))
training_steps = 1000

x = T.dmatrix("x")
y = T.dvector("y")
w1 = shared(numpy.random.randn(features, hidden), name="w1")
b1 = shared(numpy.zeros(hidden), name="b1")
w2 = shared(numpy.random.randn(hidden), name="w2")
b2 = shared(0., name="b2")
p1 = T.tanh(T.dot(x, w1) + b1)
p2 = T.tanh(T.dot(p1, w2) + b2)
prediction = p2 > 0.5
epsilon = 1e-7
p2_clipped = T.clip(p2, epsilon, 1.0 - epsilon)
error = - (y * T.log(p2_clipped) + (1 - y) * T.log(1 - p2_clipped))

loss = error.mean() + 0.01 * (T.sum(w1**2) + T.sum(w2**2))


gw1, gb1, gw2, gb2 = T.grad(loss, [w1, b1, w2, b2])

train = function(inputs=[x,y],outputs=[p2, error], updates=((w1, w1 - 0.1 * gw1),
(b1, b1 - 0.1 * gb1), (w2, w2 - 0.1 * gw2), (b2, b2 - 0.1 * gb2)))

predict = function(inputs=[x], outputs=[prediction])

print("Accuracy before Training:", sklearn.metrics.accuracy_score(D[1], numpy.array(predict(D[0])).ravel()))
for i in range(training_steps):
    prediction, error = train(D[0], D[1])
print("Accuracy after Training:", sklearn.metrics.accuracy_score(D[1],numpy.array(predict(D[0])).ravel()))

Accuracy before Training: 0.518
Accuracy after Training: 0.725


In [27]:
from math import exp
from random import seed
from random import random

# Initialize a network
def initialize_network(n_inputs, n_hidden, n_outputs):
    network = list()
    hidden_layer = [{'weights':[random() for i in range(n_inputs + 1)]} for i in range(n_hidden)]
    network.append(hidden_layer)
    output_layer = [{'weights':[random() for i in range(n_hidden + 1)]} for i in range(n_outputs)]
    network.append(output_layer)
    return network

# Calculate neuron activation for an input
def activate(weights, inputs):
    activation = weights[-1]
    for i in range(len(weights)-1):
        activation += weights[i] * inputs[i]
    return activation

# Transfer neuron activation
def transfer(activation):
    return 1.0 / (1.0 + exp(-activation))

# Forward propagate input to a network output
def forward_propagate(network, row):
    inputs = row
    for layer in network:
        new_inputs = []
        for neuron in layer:
            activation = activate(neuron['weights'], inputs)
            neuron['output'] = transfer(activation)
            new_inputs.append(neuron['output'])
        inputs = new_inputs
    return inputs

# Calculate the derivative of an neuron output
def transfer_derivative(output):
    return output * (1.0 - output)

# Backpropagate error and store in neurons
def backward_propagate_error(network, expected):
    for i in reversed(range(len(network))):
        layer = network[i]
        errors = list()
        if i != len(network)-1:
            for j in range(len(layer)):
                error = 0.0
                for neuron in network[i + 1]:
                    error += (neuron['weights'][j] * neuron['delta'])
                errors.append(error)
        else:
            for j in range(len(layer)):
                neuron = layer[j]
                errors.append(expected[j] - neuron['output'])
        for j in range(len(layer)):
            neuron = layer[j]
            neuron['delta'] = errors[j] * transfer_derivative(neuron['output'])

# Update network weights with error
def update_weights(network, row, l_rate):
    for i in range(len(network)):
        inputs = row[:-1]
        if i != 0:
            inputs = [neuron['output'] for neuron in network[i - 1]]
        for neuron in network[i]:
            for j in range(len(inputs)):
                neuron['weights'][j] += l_rate * neuron['delta'] * inputs[j]
            neuron['weights'][-1] += l_rate * neuron['delta']

#Train a network for a fixed number of epochs
def train_network(network, train, l_rate, n_epoch, n_outputs):
    for epoch in range(n_epoch):
        sum_error = 0
        for row in train:
            outputs = forward_propagate(network, row)
            expected = [0 for i in range(n_outputs)]
            expected[row[-1]] = 1
            sum_error += sum([(expected[i]-outputs[i])**2 for i in range(len(expected))])
            backward_propagate_error(network, expected)
            update_weights(network, row, l_rate)
        print('>epoch=%d, lrate=%.3f, error=%.3f' % (epoch, l_rate, sum_error))

#Test training backprop algorithm
seed(1)
dataset = [[2.7810836,2.550537003,0],
    [1.465489372,2.362125076,0],
    [3.396561688,4.400293529,0],
    [1.38807019,1.850220317,0],
    [3.06407232,3.005305973,0],
    [7.627531214,2.759262235,1],
    [5.332441248,2.088626775,1],
    [6.922596716,1.77106367,1],
    [8.675418651,-0.242068655,1],
    [7.673756466,3.508563011,1]]
n_inputs = len(dataset[0]) - 1
n_outputs = len(set([row[-1] for row in dataset]))
network = initialize_network(n_inputs, 2, n_outputs)
train_network(network, dataset, 0.5, 20, n_outputs)
for layer in network:
    print(layer)

>epoch=0, lrate=0.500, error=6.350
>epoch=1, lrate=0.500, error=5.531
>epoch=2, lrate=0.500, error=5.221
>epoch=3, lrate=0.500, error=4.951
>epoch=4, lrate=0.500, error=4.519
>epoch=5, lrate=0.500, error=4.173
>epoch=6, lrate=0.500, error=3.835
>epoch=7, lrate=0.500, error=3.506
>epoch=8, lrate=0.500, error=3.192
>epoch=9, lrate=0.500, error=2.898
>epoch=10, lrate=0.500, error=2.626
>epoch=11, lrate=0.500, error=2.377
>epoch=12, lrate=0.500, error=2.153
>epoch=13, lrate=0.500, error=1.953
>epoch=14, lrate=0.500, error=1.774
>epoch=15, lrate=0.500, error=1.614
>epoch=16, lrate=0.500, error=1.472
>epoch=17, lrate=0.500, error=1.346
>epoch=18, lrate=0.500, error=1.233
>epoch=19, lrate=0.500, error=1.132
[{'weights': [-1.4688375095432327, 1.850887325439514, 1.0858178629550297], 'output': 0.029980305604426185, 'delta': -0.0059546604162323625}, {'weights': [0.37711098142462157, -0.0625909894552989, 0.2765123702642716], 'output': 0.9456229000211323, 'delta': 0.0026279652850863837}]
[{'weights